# Demo 5 of 6 - Data quality and reconciliation

**Why we are doing this.** Kenny's complaint: validating incoming data, identifying duplicate or invalid records, and reconciling source to target results.

**What we will prove.**
1. Quality rules can be attached directly to a table, run on a schedule, and checked against expectations (data metric functions).
2. Bad rows are kept with a reason, never silently dropped (quarantine).
3. For every file we can show rows received, rows loaded, rows rejected, rows valid and rows quarantined, and whether they add up (reconciliation).

**Before you start:** the labs JSON file from Demo 2, Step 5, must have been loaded.

**How to read this notebook.** Every cell has one of four labels:
- **SETUP - ALREADY DONE (read only):** code that already exists in the account. It is shown so the engineers can see how it was built. You do not run it.
- **RUN DURING DEMO:** a SQL cell you run live. Click inside it and press the run button (or Cmd+Enter).
- **UI STEP:** something you do in the Snowsight screens or the AWS console, not in the notebook.
- **SAY THIS:** your talk track for that moment.

### SETUP - ALREADY DONE (read only, do not run)
**A custom quality rule is just SQL**

```sql
CREATE DATA METRIC FUNCTION SOMATUS_DEMO.OPS.DMF_INVALID_EGFR(T TABLE(TEST VARCHAR, RESULT_VALUE FLOAT))
RETURNS NUMBER AS
'SELECT COUNT_IF(TEST = ''eGFR'' AND (RESULT_VALUE < 0 OR RESULT_VALUE > 150)) FROM T';
-- Also attached: built-in NULL_COUNT, ROW_COUNT, FRESHNESS, and a custom duplicate-key check DMF_DUP_LAB_KEY.
-- Each one is attached with an expectation, for example: VALUE = 0, on a 5-minute schedule.
```

### SETUP - ALREADY DONE (read only, do not run)
**Quarantine: every row is tagged with a reason, then split into valid and quarantined**

```sql
CREATE DYNAMIC TABLE SOMATUS_DEMO.SILVER.LAB_RESULTS_CHECKED TARGET_LAG='1 minute' WAREHOUSE=WH_TRANSFORM AS
SELECT l.*,
       CASE WHEN MEMBER_ID IS NULL THEN 'MISSING_MEMBER_ID'
            WHEN TEST = 'eGFR' AND (RESULT_VALUE < 0 OR RESULT_VALUE > 150) THEN 'INVALID_EGFR'
            WHEN ROW_NUMBER() OVER (PARTITION BY LAB_ID, TEST ORDER BY LOADED_AT) > 1 THEN 'DUPLICATE'
       END AS REJECT_REASON
FROM SOMATUS_DEMO.SILVER.LAB_RESULTS l;
-- LAB_RESULTS_VALID      = rows where REJECT_REASON IS NULL  (feeds Gold)
-- LAB_RESULTS_QUARANTINE = rows where REJECT_REASON IS NOT NULL  (for review)
```

### RUN DURING DEMO
Set the role, warehouse and database. Run this first, every time you open the notebook.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_TRANSFORM;

## Step 1 - Quality rules attached to the table

### RUN DURING DEMO
Show the custom quality rules.

In [ ]:
%%sql -r dmf_list
SHOW DATA METRIC FUNCTIONS IN SCHEMA SOMATUS_DEMO.OPS;

### RUN DURING DEMO
Show which rules are attached to the Silver lab results table, on which columns, and on what schedule.

In [ ]:
%%sql -r dmf_attached
SELECT METRIC_NAME, REF_ARGUMENTS, SCHEDULE, SCHEDULE_STATUS
FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
       REF_ENTITY_NAME => 'SOMATUS_DEMO.SILVER.LAB_RESULTS', REF_ENTITY_DOMAIN => 'TABLE'));

### RUN DURING DEMO
Run all the checks now. EXPECTATION_VIOLATED = TRUE means the rule failed: for example, there are invalid eGFR values or duplicate lab records in the client file.

In [ ]:
%%sql -r expectations
SELECT METRIC_NAME, EXPECTATION_NAME, EXPECTATION_EXPRESSION, VALUE, EXPECTATION_VIOLATED
FROM TABLE(SYSTEM$EVALUATE_DATA_QUALITY_EXPECTATIONS(REF_ENTITY_NAME => 'SOMATUS_DEMO.SILVER.LAB_RESULTS'));

### UI STEP - do this in the screen, not in the notebook
**Show the same checks in the screen**

1. In the left menu, click **Data**, then **Databases**.
2. Open **SOMATUS_DEMO > SILVER > Dynamic Tables** and click **LAB_RESULTS**.
3. Click the **Data Quality** tab. Each rule appears with its latest value and whether it passed. (Results appear after the first 5-minute scheduled run.)

### RUN DURING DEMO
Show the history of scheduled check results. This is what you would alert on or chart over time.

In [ ]:
%%sql -r dq_history
SELECT MEASUREMENT_TIME, METRIC_NAME, VALUE
FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
WHERE TABLE_DATABASE = 'SOMATUS_DEMO' AND TABLE_NAME = 'LAB_RESULTS'
ORDER BY MEASUREMENT_TIME DESC LIMIT 20;

> **SAY THIS:** Quality checks live with the table, run on a schedule, and are visible to anyone who opens the table. No separate tool.

## Step 2 - Bad rows are quarantined with a reason

### RUN DURING DEMO
Count quarantined rows by reason.

In [ ]:
%%sql -r quarantine_summary
SELECT REJECT_REASON, COUNT(*) AS ROW_COUNT
FROM SOMATUS_DEMO.SILVER.LAB_RESULTS_QUARANTINE GROUP BY 1 ORDER BY 2 DESC;

### RUN DURING DEMO
Show sample quarantined rows, each with the file it came from.

In [ ]:
%%sql -r quarantine_rows
SELECT LAB_ID, MEMBER_ID, TEST, RESULT_VALUE, REJECT_REASON, FILE_NAME
FROM SOMATUS_DEMO.SILVER.LAB_RESULTS_QUARANTINE LIMIT 10;

> **SAY THIS:** Nothing is silently dropped. Gold only sees valid rows, and the data team has a review queue with a reason for every rejected row.

## Step 3 - Source to target reconciliation, per file

### RUN DURING DEMO
For each lab file: rows in the file, rows loaded, rows rejected at load, Bronze records, expected test rows after flattening, Silver rows, valid rows and quarantined rows. STATUS is RECONCILED when every stage adds up.

In [ ]:
%%sql -r recon
SELECT * FROM SOMATUS_DEMO.OPS.RECON_LAB_FILES;

### RUN DURING DEMO
The same file-level view for member files. The bad CIGNA file from Demo 2 shows 100 in the file, 90 loaded and 10 rejected.

In [ ]:
%%sql -r recon_members
SELECT FILE_NAME, STATUS, ROW_PARSED AS ROWS_IN_FILE, ROW_COUNT AS LOADED, ERROR_COUNT AS REJECTED, FIRST_ERROR_MESSAGE
FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.COPY_HISTORY(
       TABLE_NAME => 'SOMATUS_DEMO.BRONZE.MEMBERS_RAW', START_TIME => DATEADD('day', -1, CURRENT_TIMESTAMP())));

> **SAY THIS:** For any file a client sends, you can answer: did it all arrive, how much was good, and where did every other row go.